# 📔 Notebook Exercise: Robust Active Learning Data Pipelines

This notebook tests your understanding of defensive coding, categorical data mapping, and precise row tracking within a `pandas` active learning pipeline.

Complete each exercise by replacing the `### YOUR CODE HERE ###` placeholders. Run the verification cells to check your work.

---

## 🛠️ Setup & Mock Data Generation

Run this cell first to set up your environment and create a messy, realistic dataset.

In [ ]:
import pandas as pd
import numpy as np

# Generate a mock dataset with missing data, mixed types, and duplicate indices to simulate real-world chaos
np.random.seed(42)
n_rows = 100

mock_df = pd.DataFrame({
    'label': np.random.choice(['spam', 'uncertain', 'not_spam', np.nan], size=n_rows, p=[0.3, 0.2, 0.4, 0.1]),
    'heuristic_confidence': np.random.choice(['high', 'High ', 'MED', 'medium', 'low', np.nan, 0.8, 0.2], size=n_rows),
    'spam_category': np.random.choice(['phishing', 'promo', 'scam', np.nan], size=n_rows, p=[0.2, 0.4, 0.3, 0.1]),
    'text': [f"Message variant {i}" for i in range(n_rows)]
})

# Scramble the index intentionally to simulate a pre-shuffled dataset
mock_df = mock_df.sample(frac=1, random_state=101)
print("Data initialized! Current Index layout:", mock_df.index[:5].tolist())
mock_df.head()


---

## 🏋️ Exercise 1: Proportional Stratified Sampler

### Instructions

Implement `stratified_sample`. It must:

1. Handle cases where `category_col` is missing — log a warning and fall back to a plain sample.
2. Fill missing values in the target category column with `'unknown'`.
3. Sample rows **proportionally**, ensuring *at least 1 row* is pulled from every existing category.
4. Return exactly $n$ rows, fully shuffled, with a reset index.

In [ ]:
def stratified_sample(df, n, category_col='spam_category'):
    """Proportional quota sample across category_col values."""
    # 1. Fallback check: if column doesn't exist, return a plain sample safely
    if category_col not in df.columns:
        print(f'[warn] {category_col!r} missing — using plain random sample')
        ### YOUR CODE HERE (approx 1 line) ###
        raise NotImplementedError()

    # 2. Handle missing data by filling NaNs with 'unknown'
    ### YOUR CODE HERE (approx 1 line) ###
    cats = None   # should be df[category_col] with NaNs filled

    # 3. Calculate fractions of each category
    fracs = cats.value_counts(normalize=True)
    parts = []

    # 4. Loop through categories and build quotas
    for cat, frac in fracs.items():
        # Ensure quota is at least 1, and doesn't exceed available rows in that chunk
        chunk = df[cats == cat]
        ### YOUR CODE HERE (approx 2 lines: calculate quota and sample chunk) ###
        quota = None
        sampled_chunk = None

        parts.append(sampled_chunk)

    # 5. Combine, shuffle, trim to exact length 'n', and clear the index
    ### YOUR CODE HERE (approx 1 line) ###
    return None


### 🧪 Verification Test 1

In [ ]:
# Test Case A: Missing Column Fallback
fallback_sample = stratified_sample(mock_df, n=10, category_col='non_existent_column')
assert len(fallback_sample) == 10, "Fallback sample length should be 10"

# Test Case B: Stratification Preservation
sample_df = stratified_sample(mock_df, n=20, category_col='spam_category')
assert len(sample_df) == 20, "Sample length must match requested n exactly"
assert 'unknown' in mock_df['spam_category'].fillna('unknown').values, "Data setup issue"
print("✅ Exercise 1 Passed Successfully!")


---

## 🏋️ Exercise 2: Priority Ranker (With Categorical Mapping & Safe Tracking)

### Instructions

Implement `rank_by_priority`. Fix the two fatal flaws:

1. **Categorical Mapping:** Convert text strings (`'high'`, `'med'`, `'low'`) to numeric scales ($1.0$, $0.5$, $0.1$). Be defensive against whitespace and mixed casing.
2. **Index Tracking Preservation:** Capture the *original data index* under a column named `_source_index` **before** sorting or resetting the index.

**Priority rules:**
- `'spam'` → add confidence score
- `'uncertain'` → add `(1 − confidence score)`
- `'not_spam'` → subtract half the confidence score

In [ ]:
def rank_by_priority(df):
    df = df.copy()

    # 1. PRESERVE INDEX: Capture original index immediately before any modifications
    ### YOUR CODE HERE ###
    df['_source_index'] = None

    # Extract raw confidence data
    if 'heuristic_confidence' in df.columns:
        raw_conf = df['heuristic_confidence']
    elif 'confidence' in df.columns:
        raw_conf = df['confidence']
    else:
        raw_conf = pd.Series(0.5, index=df.index)

    # 2. CATEGORICAL MAPPING & COERCION
    if raw_conf.dtype == 'object' or isinstance(raw_conf.iloc[0], str):
        # Normalize casing and strip trailing spaces
        clean_conf = raw_conf.astype(str).str.lower().str.strip()

        # Build your mapping dictionary
        ### YOUR CODE HERE (Define mapping dictionary) ###
        confidence_map = {}

        # Map values and fill remaining NaNs with a neutral baseline 0.5
        conf = clean_conf.map(confidence_map).fillna(0.5)
    else:
        # If already numeric, safely coerce non-numeric remnants to NaN, fill with 0.5
        conf = pd.to_numeric(raw_conf, errors='coerce').fillna(0.5)

    # Standardize label strings
    label = df.get('label', pd.Series('unknown', index=df.index)).astype(str).str.lower()

    # 3. PRIORITY ALGORITHM MATH
    pri = pd.Series(0.0, index=df.index)
    ### YOUR CODE HERE ###
    # Rules:
    #   - label == 'spam'     → pri += conf
    #   - label == 'uncertain'→ pri += (1 - conf)
    #   - label == 'not_spam' → pri -= (conf * 0.5)

    df['_priority'] = pri

    # Sort descending by priority, reset the outer index securely, and return
    return df.sort_values('_priority', ascending=False).reset_index(drop=True)


### 🧪 Verification Test 2

In [ ]:
ranked_df = rank_by_priority(mock_df)

# Check tracking integrity
assert '_source_index' in ranked_df.columns, "You forgot to preserve '_source_index' as a permanent column!"
assert ranked_df.loc[0, '_source_index'] in mock_df.index, "The saved source index values don't match the original source dataframe indices!"

# High confidence spam should rise near the top
top_row = ranked_df.iloc[0]
bottom_row = ranked_df.iloc[-1]

assert top_row['_priority'] >= bottom_row['_priority'], "Sorting validation mismatch"
print("✅ Exercise 2 Passed Successfully!")


---

## 🏋️ Exercise 3: Complete Active Learning Wave Selector Pipeline

### Instructions

Assemble the full pipeline. `select_wave` must:

1. Look up the target batch size from `wave_sizes_dict` using `wave_id`.
2. Run `rank_by_priority` on the data pool.
3. Exclude rows whose `_source_index` is in `already_done_set`.
4. Return the top `target` rows with a clean reset index.

In [ ]:
def select_wave(df, wave_id, already_done_set, wave_sizes_dict):
    """Filters out already completed tasks, prioritizing the next payload slice."""
    # 1. Extract target volume size from dictionary using wave_id
    ### YOUR CODE HERE ###
    target = None

    # 2. Prioritize data pool
    ### YOUR CODE HERE ###
    ranked = None

    # 3. Exclude values that exist within already_done_set using the permanent tracking column
    ### YOUR CODE HERE ###
    filtered_ranked = None

    # 4. Snip the head to size and return clean index output
    return filtered_ranked.head(target).reset_index(drop=True)


### 🧪 Verification Test 3

In [ ]:
# Setup simulation parameters
WAVE_SIZES = {1: 15, 2: 30}
already_processed_indices = {42, 11, 88, 99, 5}  # Simulating IDs already saved to a CSV file

wave_1_batch = select_wave(
    mock_df,
    wave_id=1,
    already_done_set=already_processed_indices,
    wave_sizes_dict=WAVE_SIZES
)

assert len(wave_1_batch) == 15, f"Expected 15 items back, got {len(wave_1_batch)}"
assert not wave_1_batch['_source_index'].isin(already_processed_indices).any(),     "Blacklisted tracking ID sneaked past your filter!"
print("🏆 All pipeline exercises completed successfully! Exceptional work!")


---

## 💡 Hints (expand if you're stuck)

<details>
<summary><b>Exercise 1 — Stratified Sampler hints</b></summary>

```python
# Fallback line:
return df.sample(n=n).reset_index(drop=True)

# Fill NaNs:
cats = df[category_col].fillna('unknown')

# Inside the loop:
quota = max(1, min(int(round(frac * n)), len(chunk)))
sampled_chunk = chunk.sample(n=quota, replace=False)

# Combine & trim:
return pd.concat(parts).sample(frac=1).head(n).reset_index(drop=True)
```
</details>

<details>
<summary><b>Exercise 2 — Priority Ranker hints</b></summary>

```python
# Preserve index:
df['_source_index'] = df.index

# Mapping dict:
confidence_map = {'high': 1.0, 'med': 0.5, 'medium': 0.5, 'low': 0.1}

# Priority logic:
pri[label == 'spam']     += conf[label == 'spam']
pri[label == 'uncertain']+= 1 - conf[label == 'uncertain']
pri[label == 'not_spam'] -= conf[label == 'not_spam'] * 0.5
```
</details>

<details>
<summary><b>Exercise 3 — Wave Selector hints</b></summary>

```python
target = wave_sizes_dict[wave_id]
ranked = rank_by_priority(df)
filtered_ranked = ranked[~ranked['_source_index'].isin(already_done_set)]
```
</details>
